# Issue #234 — canonical source registry, identity and temporal store schema

**Contract:** BJ-UNIFIED-DATA-SOURCE-SCHEMA-001 / v1 (PI-user design freeze, 2026-08-16)
**Audit mode:** candidate
**Owner role:** Engineer · Project: BJ Data Platform · Milestone: M1 · Parent: #232

This notebook is a thin orchestration layer: it runs the reusable
`src/source_schema` package top-to-bottom, renders every committed
machine-readable table and verifies the immutable package. All durable
evidence lives in `results/bj234_source_schema_v1/`.

What is frozen here:
- source registry (every source gets admissible-use class with evidence, REQ-DS01)
- canonical fixture identity (unique within `issue_code x match_idx`, REQ-DS02)
- decision-epoch / observed / received / kickoff semantics (REQ-DS03)
- normalized tables and key/domain constraints rejecting duplicate/invalid M×3 cells (REQ-DS04)
- inference / proxy / settlement namespaces physically distinguishable (REQ-DS05)
- SQLite WAL + pandas/PyArrow + measured scale/concurrency justification (REQ-DS06)
- notebook tables bind machine-readable outputs and source hashes (REQ-DS07)
- handoff names exact interfaces for #235/#236/#237/#238 (REQ-DS08)


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path("/home/drwho1985/beijing-lot-234")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import pyarrow as pa

from source_schema import contract, crosswalk, feasibility, profiler, publication, schemas


## 1. Profile every existing source database (read-only)

The issue contract names four databases plus the #203/#214/#229 immutable
packages.  We profile each: tables, row counts, keys, timestamp ranges,
nulls, duplicates and journal mode.  All probes are read-only
(`PRAGMA query_only=ON`).


In [2]:
profile = profiler.profile_all()
rows = []
for db, info in profile.items():
    if not info.get("present"):
        rows.append({"database": db, "size_mb": None, "journal_mode": None,
                     "tables": "MISSING", "identity_dup": None})
        continue
    rows.append({
        "database": db,
        "size_mb": info.get("size_mb"),
        "journal_mode": info.get("journal_mode"),
        "tables": ",".join(t["name"] for t in info.get("tables", [])),
        "identity_dup": info.get("identity_dup"),
    })
pd.DataFrame(rows)


,database,size_mb,journal_mode,tables,identity_dup
0,beidan_xlsm.sqlite,6872.9,delete,"action_candidates,action_definitions,exotic_od...",1
1,beidan_features.db,47.2,delete,matches,ERR
2,beidan_history.db,6.8,delete,"historical_matches,scrape_log",0
3,beidan_live.db,261.8,wal,"matches,settlements,snapshots",0
4,500_bjdc.sqlite,17031.8,wal,"adhoc_betting_request_ledger,fetch_log,issues,...",None


## 2. Timestamp clock probes — the causal-admissibility evidence

The critical finding: `beidan_xlsm.sqlite` `market_snapshots.snapshot` holds
*labels* (`initial` / `10h` / `1h` / `final`), not exact timestamps; the
`beidan_history.db` settlement table has no settled-timestamp column; and
`beidan_features.db` kickoff formats are mixed (time-only `08:00:00` strings
sort before dates).  Only `beidan_live.db` carries native ISO-8601 `Z`
snapshot timestamps prospectively from 2026-05.


In [3]:
clocks = profiler.classify_source_clocks(profile)
pd.DataFrame(clocks)[["source", "family", "admissible_use", "clock_semantics"]]


,source,family,admissible_use,clock_semantics
0,beidan_live.db,F1_BJ_FIRST_PARTY,STRICT_CAUSAL,NATIVE_PROVEN
1,beidan_features.db,F2_INTERNAL_ARCHIVE,RETROSPECTIVE_PROXY,LABELED_RELATIVE
2,beidan_history.db,F2_INTERNAL_ARCHIVE,SETTLEMENT_ONLY,ABSENT
3,beidan_xlsm.sqlite (market_snapshots),F2_INTERNAL_ARCHIVE,RETROSPECTIVE_PROXY,LABELED_RELATIVE
4,beidan_xlsm.sqlite (official_500_settlements),F2_INTERNAL_ARCHIVE,SETTLEMENT_ONLY,FILE_MTIME
5,beidan_xlsm.sqlite (identity),F2_INTERNAL_ARCHIVE,RETROSPECTIVE_PROXY,ABSENT


In [4]:
# Exact timestamp ranges from the probe (machine-readable evidence)
for db, info in profile.items():
    if not info.get("present"):
        continue
    print(f"--- {db}")
    for key in ("kickoff_range", "snapshot_range", "settled_range",
                "snapshot_labels", "settle_range_endtime", "settle_fetched_range"):
        if info.get(key):
            print(f"  {key}: {info[key]}")


--- beidan_xlsm.sqlite
  kickoff_range: ('1899-12-30 08:00:00', '2025-07-13 03:00:00')
  snapshot_labels: [('initial', 12887754), ('final', 12887754), ('1h', 12887754), ('10h', 12887754)]
  settle_range_endtime: ('2013-12-28 16:35', '2017-12-26 02:50')
  settle_fetched_range: ('2026-06-26 12:10:03', '2026-06-26 12:14:27')
--- beidan_features.db
  kickoff_range: ('08:00:00', '2025-04-15T08:30:00')
--- beidan_history.db
  kickoff_range: ('2024-01-01 14:00:00', '2026-12-31 16:00:00')
--- beidan_live.db
  kickoff_range: ('2026-04-30T19:30:00', '2026-08-18T11:06:00')
  snapshot_range: ('2026-05-02T00:57:41Z', '2026-08-16T04:23:46Z')
  settled_range: ('2026-05-09 20:06:14', '2026-08-16 04:24:13')
--- 500_bjdc.sqlite


## 3. Versioned SQL DDL + PyArrow schemas + domain dictionary

The DDL is the frozen interface for all #232 recovery branches.  It is
rendered deterministically from `src/source_schema/contract.py`; the SQL and
PyArrow schemas therefore cannot drift apart.  Physical namespaces:
`capture` (append-only control tables), `inference` (market observations /
quotes / probability estimates) and a physically separate `settlement` table.


In [5]:
ddl = schemas.render_ddl()
print(ddl[:1200])
print("...")
print(f"DDL sha256: {schemas.render_ddl_sha256()}")


-- BJ-UNIFIED-DATA-SOURCE-SCHEMA-001 / v1 (frozen 2026-08-16)
-- Canonical source registry, identity and temporal store schema.
-- Rendered deterministically from src/source_schema/contract.py.
-- SQLite WAL mode; one-writer append-only capture/control tables;
-- research snapshots are immutable partitioned Parquet (see PyArrow schemas).
-- All timestamps are timezone-aware UTC (ISO-8601 with Z).
-- outcome_code domain is exactly ('3','1','0') after explicit mapping.
-- The Beijing 0.65 coefficient is ABSENT from all source/quote/probability
-- tables; it exists only in downstream ticket settlement metadata.

PRAGMA journal_mode = WAL;
PRAGMA synchronous = NORMAL;
PRAGMA foreign_keys = ON;

-- namespace: capture
CREATE TABLE IF NOT EXISTS source_registry (
  source_id TEXT NOT NULL PRIMARY KEY,
  provider TEXT NOT NULL,
  endpoint TEXT,
  acquisition_method TEXT NOT NULL,
  terms_status TEXT NOT NULL,
  timezone TEXT NOT NULL,
  clock_semantics TEXT NOT NULL,
  cadence_seconds INTEGER,

In [6]:
# PyArrow schema for one canonical table materializes cleanly
quote_schema = schemas.arrow_schema("market_quote")
quote_schema


quote_id: string not null
observation_id: string not null
bookmaker_id: string
outcome_code: string not null
decimal_odds: double not null
quote_status: string not null

In [7]:
domain = pd.DataFrame(schemas.domain_dictionary())
domain[domain["column"].isin(["outcome_code", "observed_at_utc", "received_at_utc",
                              "kickoff_at_utc", "decision_at_utc", "fixture_id"])]


,table,column,sqlite_type,arrow_type,nullable,constraints,domain_notes
18,ingestion_batch,received_at_utc,TEXT,"timestamp[us, tz=UTC]",false,,
43,fixture,fixture_id,TEXT,string,false,PK,canonical identity; unique within issue_code x...
50,fixture,kickoff_at_utc,TEXT,"timestamp[us, tz=UTC]",false,,scheduled kickoff (UTC); NOT a decision epoch ...
56,decision_epoch,decision_at_utc,TEXT,"timestamp[us, tz=UTC]",false,,decision cutoff; must be < kickoff_at_utc for ...
60,market_observation,fixture_id,TEXT,string,false,,
64,market_observation,observed_at_utc,TEXT,"timestamp[us, tz=UTC]",false,,exact source observation time; NOT interchange...
65,market_observation,received_at_utc,TEXT,"timestamp[us, tz=UTC]",false,,trusted fetch/receive clock; NOT interchangeab...
71,market_quote,outcome_code,TEXT,string,false,,exactly 3/1/0 after explicit mapping (REQ-DS04)
75,probability_estimate,fixture_id,TEXT,string,false,,
80,probability_estimate,outcome_code,TEXT,string,false,,


## 4. Identity crosswalk (REQ-DS02)

Every legacy database binds to the canonical fixture identity through an
explicit crosswalk — never a silent fuzzy match.  `fixture_id` is unique
within `issue_code x match_idx`.


In [8]:
cw = pd.DataFrame(crosswalk.identity_crosswalk_rows())
cw


,db,table,source_key_columns,canonical_binding,resolvable,note
0,beidan_live.db,matches,"[issue, match_idx]","issue_code=issue, match_idx=match_idx → fixtur...",YES,issue x match_idx UNIQUE verified (dup=0); mat...
1,beidan_live.db,settlements,"[issue, match_idx]",settlement.fixture_id = canonical fixture,YES,settled_ts present; settlement namespace only.
2,beidan_history.db,historical_matches,"[issue, match_idx]","issue_code=issue, match_idx=match_idx → fixtur...",YES,UNIQUE verified (dup=0); no settled timestamp ...
3,beidan_features.db,matches,"[issue, match_idx]","issue_code=issue, match_idx=match_idx → fixtur...",PARTIAL,table has NO match_idx column — identity must ...
4,beidan_xlsm.sqlite,matches,"[issue, match_idx]","issue_code=issue, match_idx=match_idx → fixtur...",PARTIAL,2560 rows with NULL issue/match_idx excluded b...
5,beidan_xlsm.sqlite,official_500_settlements,"[issue, match_idx]",settlement.fixture_id = canonical fixture,YES,"settlement namespace; fetched 2026-06-26, end_..."


In [9]:
print(json.dumps(crosswalk.canonical_identity_rule(), indent=1))


{
 "rule": "fixture_id = fix_{issue_code}_{match_idx}_{sha256(issue:match)[:16]}",
 "unique_within": "issue_code x match_idx",
 "alias_requirement": "every source key binds via alias tables; silent fuzzy matching prohibited",
 "identity_status": [
  "CANONICAL",
  "ALIAS_RESOLVED",
  "UNRESOLVED"
 ],
 "example": "fix_25041_1_15302337593b03c6"
}


## 5. Source feasibility matrix (REQ-DS01) + migration-free replacement plan

Each candidate source carries acquisition cost / access / licensing /
coverage / clock grade and a concrete selected source per downstream branch
(#235 Beijing SP, #236 EU odds, #237 power features, #238 materializer).


In [10]:
fm = pd.DataFrame(feasibility.feasibility_matrix())
fm[["source_id", "family", "cost", "access", "licensing", "coverage",
    "clock_grade", "admissible_use", "selected_for"]]


,source_id,family,cost,access,licensing,coverage,clock_grade,admissible_use,selected_for
0,BJ_OFFICIAL_INSTANT_SP,F1_BJ_FIRST_PARTY,free,A,C,A,A,STRICT_CAUSAL,BJ_SP_TIMESERIES #235 (primary)
1,BEIDAN_LIVE_DB,F1_BJ_FIRST_PARTY,free,A,A,B,A,STRICT_CAUSAL,BJ_SP_TIMESERIES #235 (2026-05+ cohort)
2,BEIDAN_XLSM_WORKBOOK,F2_INTERNAL_ARCHIVE,free,A,A,A,C,RETROSPECTIVE_PROXY,EU_ODDS #236 (proxy cohort only; strict requir...
3,BEIDAN_FEATURES_DB,F2_INTERNAL_ARCHIVE,free,A,A,B,C,RETROSPECTIVE_PROXY,EU_ODDS #236 (proxy cohort)
4,BEIDAN_HISTORY_DB,F2_INTERNAL_ARCHIVE,free,A,A,B,U,SETTLEMENT_ONLY,settlement cross-check only
5,ODDS_API,F3_EU_TIMESTAMPED,paid (user authority required for purchase),B,A,C,A,STRICT_CAUSAL,EU_ODDS #236 (strict candidate — verify covera...
6,FOOTBALL_DATA_CSV,F4_FOOTBALL_DATA_CSV,free,A,B,A,C,RETROSPECTIVE_PROXY,POWER_FEATURES #237 (secondary); EU proxy #236
7,INTERNAL_RESULTS_POWER,F5_RESULTS_POWER,free,A,A,B,A,STRICT_CAUSAL,POWER_FEATURES #237 (primary)


In [11]:
print(feasibility.REPLACEMENT_PLAN["principle"])
for legacy in feasibility.REPLACEMENT_PLAN["replaced_schemas"]:
    print(f"- {legacy['legacy']} -> {legacy['replacement']}")


AGENTS.md principle 1 — no backward-compat tax. Obsolete ambiguous schemas are removed from the accepted interface, not wrapped in compatibility layers.
- beidan_xlsm.sqlite market_snapshots.snapshot column -> market_observation.observed_at_utc (exact) OR snapshot_kind='PROXY_LABEL' with NULL observed_at and explicit proxy namespace
- beidan_features.db matches.kickoff mixed formats -> fixture.kickoff_at_utc (ISO-8601 Z, UTC)
- beidan_history.db historical_matches (no settled ts) -> settlement.settled_at_utc + result_code + source_id
- beidan_xlsm.sqlite matches NULL issue/match_idx (2560 rows) -> fixture requires issue_code+match_idx NOT NULL with UNIQUE constraint; unresolved rows excluded
- probability vectors stored as wide columns (p_close_3/1/0, sp_3/1/0) -> market_quote / probability_estimate long-form outcome_code-scalar rows with UNIQUE (observation, outcome_code)


## 6. Immutable package + manifest binding (REQ-DS07)

The build publishes an immutable package with `MANIFEST.json` +
`SHA256SUMS.json` binding every machine-readable artifact and the reviewed
#203/#214/#229 source hashes.  Verification re-hashes the whole package and
re-checks the bound source hashes against the referenced immutable packages.


In [12]:
PKG = ROOT / "results/bj234_source_schema_v1"
verdict = publication.verify_package(PKG)
print(f"package verified: {verdict['files']} files")
manifest = json.loads((PKG / "MANIFEST.json").read_text())
print(f"contract: {manifest['contract_version']}")
print(f"disposition: {manifest['disposition']}")
print("bound source hashes:")
for rel, h in manifest["source_hashes"].items():
    print(f"  {rel}: {h[:16]}…" if h else f"  {rel}: (absent)")


package verified: 13 files
contract: BJ-UNIFIED-DATA-SOURCE-SCHEMA-001/v1
disposition: FROZEN_CONTRACT
bound source hashes:
  results/bj203_causal_tensor_v2/MANIFEST.json: 52051e510f291947…
  results/bj203_causal_tensor_v2/schema.json: bc3c05669fd5b625…
  results/bj214_decision_epoch_tensor_v1/MANIFEST.json: d529dfbbe1a1aada…
  results/bj214_decision_epoch_tensor_v1/schema.json: e6fce49bd9e95a3e…
  results/bj229_divergence_profile_v1/MANIFEST.json: 59d476e367c8d939…
  results/bj229_divergence_profile_v1/SHA256SUMS.json: a3e00141be882e10…


In [13]:
# Package census — every artifact the contract commits to
for p in sorted(PKG.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(PKG)}  ({p.stat().st_size} B)")


  MANIFEST.json  (4685 B)
  README.md  (1216 B)
  SHA256SUMS.json  (1100 B)
  contract.json  (2214 B)
  domain_dictionary.csv  (5346 B)
  identity_crosswalk.csv  (1448 B)
  identity_rule.json  (346 B)
  profile.json  (81145 B)
  pyarrow_schemas.json  (11940 B)
  replacement_plan.json  (3645 B)
  schema.sql  (6608 B)
  source_clock_classification.csv  (1974 B)
  source_feasibility_matrix.csv  (3144 B)


## 7. Concurrency / scale justification (REQ-DS06)

The design reuses the existing mature stack: SQLite WAL for one-writer
append-only capture/control tables, immutable partitioned Parquet for
research snapshots read by pandas/PyArrow.  No new service/database until
measured concurrency requires it.  Measured scale: `beidan_live.db` = 262 MB /
1.49M snapshots today; `beidan_xlsm.sqlite` = 6.9 GB / 51.5M snapshots as
read-only history.  Both fit the WAL + Parquet split.


In [14]:
scale = feasibility.REPLACEMENT_PLAN["concurrency_justification"]
print(f"engine: {scale['engine']}")
print(f"rationale: {scale['rationale']}")
for k, v in scale["scale_measurements"].items():
    print(f"  {k}: {v}")


engine: SQLite WAL
rationale: one-writer append-only capture/control tables; WAL allows concurrent readers during writes; measured 262 MB live DB with 1.49M snapshots operates fine today
  beidan_live_db_size_mb: 262
  snapshot_rows: 1494539
  xlsm_snapshot_rows: 51551016
  decision: Do not introduce a new service/database until measured concurrency requires it (issue contract).


## 8. Handoff to downstream subissues (REQ-DS08)

Exact interfaces named for #235 (BJ SP timeseries), #236 (EU odds), #237
(power features) and #238 (materializer).  See `replacement_plan.json` and
`source_feasibility_matrix.csv` in the immutable package; the DDL in
`schema.sql` is the contract every recovery branch must consume.


In [15]:
binding = feasibility.REPLACEMENT_PLAN["acceptance_binding"]
for req, how in binding.items():
    print(f"{req}: {how}")


REQ-DS01: source_feasibility_matrix.csv (classification with evidence)
REQ-DS02: fixture UNIQUE(issue_code, match_idx) + identity_crosswalk.csv
REQ-DS03: market_observation.observed_at_utc vs received_at_utc vs fixture.kickoff_at_utc vs decision_epoch.decision_at_utc distinct columns with domain dictionary notes
REQ-DS04: market_quote UNIQUE(observation_id, outcome_code) + CHECK outcome_code IN ('3','1','0')
REQ-DS05: physical namespaces: capture / inference / settlement tables separate; settlement table physically separate
REQ-DS06: SQLite WAL + PyArrow; scale/concurrency justification above
REQ-DS07: MANIFEST.json + SHA256SUMS.json binding machine-readable outputs and source hashes
REQ-DS08: handoff section in replacement_plan.md names exact interfaces for #235/#236/#237/#238
